In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
%cd ..
%cd ..

In [ ]:
import numpy as np
import healpy as hp
import h5py
from DVA.DVA_compute import aberration_sh_rotator, planar_orbit_velocity_direction
from scipy.spatial.transform import Rotation

from DVA import Rotation_Functions as rf
from DVA import ObservingField
from DVA import Make_ell2_GW_Background as GWB
import scipy

In [ ]:
plt.rcParams.update({'font.size': 14})

In [ ]:
#Generate Roman fields, populated with random stars.

num_stars = 1024
stars_per_side = 32
test_field0 = ObservingField.ObservingField(np.array([90.1 + 0.53/2.,0])*np.pi/180., 0.53*np.pi/180.)
#test_field0.generate_random_stars_uniform(num_stars=num_stars)
test_field0.generate_stars_uniform(stars_per_side=stars_per_side)

test_field1 = ObservingField.ObservingField(np.array([90.1 + 0.6 + 0.53/2.,360. -3*.53])*np.pi/180., 0.53*np.pi/180.)
#test_field1.generate_random_stars_uniform(num_stars=num_stars)
test_field1.generate_stars_uniform(stars_per_side=stars_per_side)

test_field2 = ObservingField.ObservingField(np.array([90.1 + 0.6 + 0.53/2.,360. -2*.53])*np.pi/180., 0.53*np.pi/180.)
#test_field2.generate_random_stars_uniform(num_stars=num_stars)
test_field2.generate_stars_uniform(stars_per_side=stars_per_side)

test_field3 = ObservingField.ObservingField(np.array([90.1 + 0.6 + 0.53/2.,360. -1*.53])*np.pi/180., 0.53*np.pi/180.)
#test_field3.generate_random_stars_uniform(num_stars=num_stars)
test_field3.generate_stars_uniform(stars_per_side=stars_per_side)

test_field4 = ObservingField.ObservingField(np.array([90.1 + 0.6 + 0.53/2.,360.])*np.pi/180., 0.53*np.pi/180.)
#test_field4.generate_random_stars_uniform(num_stars=num_stars)
test_field4.generate_stars_uniform(stars_per_side=stars_per_side)

test_field5 = ObservingField.ObservingField(np.array([90.1 + 0.6 + 0*50 + 0.53/2.,360. +.53])*np.pi/180., 0.53*np.pi/180.)
#test_field5.generate_random_stars_uniform(num_stars=num_stars)
test_field5.generate_stars_uniform(stars_per_side=stars_per_side)

fields = [test_field0, test_field1, test_field2, test_field3, test_field4, test_field5]

In [ ]:
#Plot starting positions of guide stars.
for field in fields:
    for ind in range(field.original_guide_stars_xyz.shape[0]):
        plt.scatter(field.original_guide_stars_xyz[ind,1], 
                field.original_guide_stars_xyz[ind,2])
plt.xlabel(r'$\hat{y}$')
plt.ylabel(r'$\hat{z}$')
plt.title('Field guide stars, Galactic coordinates.')
plt.show()

In [ ]:
#Generate ell=2 vector spherical harmonics at Roman star positions.

gw_sim_field_stars = []
gw_sim_guide_stars = []
for field in fields:
    gw_sim_field_stars.append(GWB.ell2_vector_harmonics(field.stars_original_positions_theta_phi))
    gw_sim_guide_stars.append(GWB.ell2_vector_harmonics(field.original_guide_stars_theta_phi))

In [ ]:
#Use Roman timing file to make array of observation times for field 0.

timing_dir = '/Users/christoa/Roman_GW_Sims/Survey_Observation_Times/'
timing_array = np.load(timing_dir + 'f146_seconds_cut_short_seasons.npy')

#Use the 5 largest timing differences as the gaps between seasons.
#These will be the gaps between the 6 season observations.
timing_diffs = np.diff(timing_array)
ordered_large_season_gaps = np.sort(timing_diffs)[-5:]
season_gaps = np.zeros(5)
season_gaps[2] = ordered_large_season_gaps[-1]
season_gaps[0:2] = ordered_large_season_gaps[0:2]
season_gaps[3:] = ordered_large_season_gaps[2:4]
season_gaps = np.round(season_gaps)
print(season_gaps)

#Each season will be 70.5 days. All 6 fields observed over 12.1 minutes.
season_length = 70.5*24*3600
cycle_length = 12.1*60
time_on_each_field = cycle_length/6
season_fullcycle_obs_times = np.arange(0, season_length, cycle_length)
print(season_fullcycle_obs_times)
season_snapshots_per_field = season_fullcycle_obs_times.size
field0_observation_times = np.zeros((6*season_snapshots_per_field))
field0_observation_times[:season_snapshots_per_field] = season_fullcycle_obs_times
for ind in range(5):
    field0_observation_times[(ind+1)*season_snapshots_per_field:(ind+2)*season_snapshots_per_field] = season_fullcycle_obs_times + np.sum(season_gaps[:ind+1])



In [ ]:
#Make array of times at which to model the GW signal.
#Ideally dense enough to sub-sample at the observing times of each field.
#Each field is re-observed every 12.1 minutes.
#Rather than sample every second, will sample every 60 seconds to avoid a huge array.
#Interpolation of this sampling should be sufficient, since power is small at high freq.
mission_time_years = 5
mission_time_seconds = mission_time_years*365*24*60*60
full_field_cadence_seconds = 60

observing_times_seconds = np.arange(0, 
            mission_time_seconds, 
                                    full_field_cadence_seconds)
observing_times_years = observing_times_seconds/(365*24*60*60)

print(observing_times_years)
print(observing_times_years.shape)

In [ ]:
#Compute DVA over a circular orbit.
orbit_object = planar_orbit_velocity_direction()
orbit_object.compute_circular_orbit_theta_phi(observing_times_years, 1, 0)
boost_magnitude = 10**-4

plt.plot(observing_times_years,
         orbit_object.circular_orbit_thetas*180./np.pi, label = 'Theta')
plt.plot(observing_times_years,
         orbit_object.circular_orbit_phis*180./np.pi, label = 'Phi')
plt.title('Orbital velocity direction, Galactic coordinates.')
plt.xlabel('Time (years)')
plt.ylabel('Angle (degrees)')
plt.legend(loc = 'lower left')
plt.show()

orbit_theta_phis = np.zeros((2, orbit_object.circular_orbit_thetas.size))
orbit_theta_phis[0,:] = orbit_object.circular_orbit_thetas
orbit_theta_phis[1,:] = orbit_object.circular_orbit_phis

print(orbit_theta_phis.shape)

orbit_direction_unit_vectors = rf.theta_phis_to_xyzs(orbit_theta_phis.T)
print(orbit_direction_unit_vectors.shape)

interp_orbit_thetas = scipy.interpolate.CubicSpline(observing_times_seconds, 
                                                orbit_object.circular_orbit_thetas)
interp_orbit_phis = scipy.interpolate.CubicSpline(observing_times_seconds, 
                                                orbit_object.circular_orbit_phis)

interp_orbit_direction_unit_vectors = scipy.interpolate.CubicSpline(observing_times_seconds, 
                                                orbit_direction_unit_vectors, axis = 0)


In [ ]:
#Model the (Black Hole Binary) Gravitatinal Wave Background.

gwb_object = GWB.gw_background(10**-14, 1./(2*full_field_cadence_seconds), 
                               1./(mission_time_seconds),
                              type = 'NANOGRAVPowerLawFit')
gwb_object.predict_full_sky_astrometric_gw_variance()
print(gwb_object.frequencies.size)
print(gwb_object.index)

In [ ]:
print('Expected RMS level of GW-induced stellar deflections is ...')
print('For deflections averaged over ' + str(full_field_cadence_seconds) + ' seconds (sets highest GW frequency)...')
print('and observing time of ' + str(mission_time_years) + ' years (sets lowest GW frequency):')
print(str(gwb_object.dn_theta_time_variance**0.5) + ' radians.')

In [ ]:
print('Confirming that sum over GW frequency variance is the same as time variance (Parseval Theorem).')
print(np.sum(gwb_object.dn_theta_freq_variance/gwb_object.dn_theta_time_variance))

In [ ]:
hc_ad_hoc_error = (6*np.pi**3*gwb_object.frequencies/gwb_object.delta_f)**0.5
hc_ad_hoc_error *= ((1e-3)*(3600*180/np.pi)**-1)*(gwb_object.delta_f/(10**8))**0.5
#print(hc_ad_hoc_error)

#print(hc_ad_hoc_error/(gwb_object.frequencies**0.5))

In [ ]:
#Plot Graviational Wave Background amplitude.

plt.plot(gwb_object.frequencies[1:], gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal')
plt.hlines(gwb_object.full_sky_astrometric_gw_variance_dn_freq**0.5, 
           gwb_object.frequencies[1],
           gwb_object.frequencies[-1], label = 'Error, full sky, unbinned',
          color = 'orange')
plt.hlines((gwb_object.full_sky_astrometric_gw_variance_dn_freq/100)**0.5, 
           gwb_object.frequencies[1],
           gwb_object.frequencies[-1], label = 'Error, full sky, binnedx100',
          color = 'green')
plt.xscale('log')
plt.yscale('log')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.xlabel('Frequencies (Hz)')
plt.legend(loc = 'lower left')
plt.title(r'RMS Deflection from BHBB in $\Delta f =$' + '{:0.1e}'.format(gwb_object.delta_f))
plt.show()

plt.plot(gwb_object.frequencies[1:], gwb_object.hc[1:], label = 'Signal')
#plt.plot(gwb_object.frequencies[1:], 
#         gwb_object.full_sky_astrometric_gw_variance_hc[1:]**0.5,
#        label = 'Error, full sky, unbinned')
plt.plot(gwb_object.frequencies[1:], 
         hc_ad_hoc_error[1:], label = 'Error')
plt.plot(gwb_object.frequencies[1:], 
         100*hc_ad_hoc_error[1:], label = 'Errorx100')
#plt.plot(gwb_object.frequencies[1:], 
#         (gwb_object.full_sky_astrometric_gw_variance_hc[1:]/10)**0.5,
#        label = 'Error, full sky, binnedx100')
plt.xscale('log')
plt.yscale('log')
plt.ylabel(r'$h_c$ (radians)')
plt.xlabel('Frequencies (Hz)')
plt.title(r'Characteristic Strain')
plt.legend(loc = 'upper left')
plt.show()

In [ ]:
#Draw a realization of the gravitational wave background.
#Model the background via 10 ell=2 E/B vector spherical harmonics.

GW_alms_freq_space = gwb_object.realize_spherical_harmonic_amplitudes_freq_space()
for ind in range(10):
    plt.plot(gwb_object.ell2_times_seconds, 
             gwb_object.ell2_time_amplitudes[ind,:])
plt.xlabel('Time (seconds)')
plt.title(r'$\ell=2$ spherical harmonic amplitudes, BHBB')
plt.show()

#Create a function to interpolate this realization, for uneven time sampling.
#Interpolation can lose some very high frequency power, but this is a small effect.
interp_a2m_amps = scipy.interpolate.CubicSpline(gwb_object.ell2_times_seconds, 
                                                gwb_object.ell2_time_amplitudes,
                                               axis = 1)

for ind in range(10):
    plt.plot(field0_observation_times, 
             interp_a2m_amps(field0_observation_times)[ind,:])
plt.xlabel('Time (seconds)')
plt.title(r'$\ell=2$ spherical harmonic amplitudes, BHBB')
plt.show()

In [ ]:
#Now simulate observation of GW background.
#Simulate with and without noise.
#Also simulate with and without DVA.

#Dimension is (# of fields, # of gradient modes, # of time samples).
magnitudes_no_DVA_no_noise_no_DVA_fitting = np.zeros((6, 4, 
                                                      field0_observation_times.size))
magnitudes_no_DVA_no_noise_with_DVA_fitting = np.zeros((6, 4, 
                                                      field0_observation_times.size))

magnitudes_with_DVA_no_noise_no_DVA_fitting = np.zeros((6, 4, 
                                                      field0_observation_times.size))
magnitudes_with_DVA_no_noise_with_DVA_fitting = np.zeros((6, 4, 
                                                      field0_observation_times.size))

magnitudes_with_DVA_with_noise_no_DVA_fitting = np.zeros((6, 4, 
                                                      field0_observation_times.size))
magnitudes_with_DVA_with_noise_with_DVA_fitting = np.zeros((6, 4, 
                                                      field0_observation_times.size))

magnitudes_with_DVA_no_noise_no_GW_no_DVA_fitting = np.zeros((6, 4, 
                                                      field0_observation_times.size))
magnitudes_with_DVA_no_noise_no_GW_with_DVA_fitting = np.zeros((6, 4, 
                                                      field0_observation_times.size))

noise_level_arcmin_per_star = 1.e-3
#noise_level_arcmin_per_star = 1.e-9
noise_level_radians_per_star = noise_level_arcmin_per_star/(3600.)*np.pi/180.

for ind_t in range(field0_observation_times.size):
    if ind_t%10000 == 0:
        print(ind_t)
    for ind in range(6):
        ell2_amplitudes = interp_a2m_amps(field0_observation_times[ind_t] + ind*time_on_each_field)
        #theta = interp_orbit_thetas(field0_observation_times[ind_t] + ind*time_on_each_field)
        #phi = interp_orbit_phis(field0_observation_times[ind_t] + ind*time_on_each_field)
        orbit_direction_xyz = interp_orbit_direction_unit_vectors(field0_observation_times[ind_t] + ind*time_on_each_field)
        orbit_direction_theta_phi = rf.xyz_to_theta_phi(orbit_direction_xyz)
        theta = orbit_direction_theta_phi[0]
        phi = orbit_direction_theta_phi[1]
        gw_sim_guide_stars[ind].generate_ell2_gw_signal(ell2_amplitudes[:5], 
                                                        ell2_amplitudes[5:])
        gw_sim_field_stars[ind].generate_ell2_gw_signal(ell2_amplitudes[:5], 
                                                        ell2_amplitudes[5:])
        fields[ind].compute_aberration_from_original_positions(boost_magnitude, 
                                                               theta,
                                                             phi, 
                                                               boost_order = 3)

        
        fields[ind].perturb_original_guide_stars(gw_sim_guide_stars[ind].gw_signal_ell2_cartesian_vector, 
                                                add_noise = False)
        fields[ind].perturb_original_field_stars(gw_sim_field_stars[ind].gw_signal_ell2_cartesian_vector, 
                                                 add_noise = False)
        fields[ind].project_onto_gradient_modes()
        magnitudes_no_DVA_no_noise_no_DVA_fitting[ind,:,ind_t] = fields[ind].diff_proj_to_grad
        magnitudes_no_DVA_no_noise_with_DVA_fitting[ind,:,ind_t] = fields[ind].diff_minus_DVA_rot_fit_proj_to_grad



        fields[ind].perturb_original_guide_stars(fields[ind].DVA_perturbation_guide_stars_xyz + gw_sim_guide_stars[ind].gw_signal_ell2_cartesian_vector,
                                                add_noise = False)
        fields[ind].perturb_original_field_stars(fields[ind].DVA_perturbation_field_stars_xyz + gw_sim_field_stars[ind].gw_signal_ell2_cartesian_vector, 
                                                 add_noise = False)
        fields[ind].project_onto_gradient_modes()
        magnitudes_with_DVA_no_noise_no_DVA_fitting[ind,:,ind_t] = fields[ind].diff_proj_to_grad
        magnitudes_with_DVA_no_noise_with_DVA_fitting[ind,:,ind_t] = fields[ind].diff_minus_DVA_rot_fit_proj_to_grad


        
        fields[ind].perturb_original_guide_stars(fields[ind].DVA_perturbation_guide_stars_xyz + gw_sim_guide_stars[ind].gw_signal_ell2_cartesian_vector, add_noise = True,
                                                noise_level=noise_level_radians_per_star)
        fields[ind].perturb_original_field_stars(fields[ind].DVA_perturbation_field_stars_xyz + gw_sim_field_stars[ind].gw_signal_ell2_cartesian_vector, add_noise = True, 
                                                 noise_level=noise_level_radians_per_star/(100.))
        fields[ind].project_onto_gradient_modes()
        magnitudes_with_DVA_with_noise_no_DVA_fitting[ind,:,ind_t] = fields[ind].diff_proj_to_grad
        magnitudes_with_DVA_with_noise_with_DVA_fitting[ind,:,ind_t] = fields[ind].diff_minus_DVA_rot_fit_proj_to_grad


        fields[ind].perturb_original_guide_stars(fields[ind].DVA_perturbation_guide_stars_xyz, add_noise = False)
        fields[ind].perturb_original_field_stars(fields[ind].DVA_perturbation_field_stars_xyz, add_noise = False)
        fields[ind].project_onto_gradient_modes()
        magnitudes_with_DVA_no_noise_no_GW_no_DVA_fitting[ind,:,ind_t] = fields[ind].diff_proj_to_grad
        magnitudes_with_DVA_no_noise_no_GW_with_DVA_fitting[ind,:,ind_t] = fields[ind].diff_minus_DVA_rot_fit_proj_to_grad



        

In [ ]:
#Plot results with no simulated DVA and no simulated noise.

rad_to_nano_as = (180/np.pi)*3600*10**9
rad_to_micro_as = (180/np.pi)*3600*10**6
rad_to_mas = (180/np.pi)*3600*10**3
rad_to_as = (180/np.pi)*3600

labels = ['XX', 'XY', 'YY', 'YX']
colors = ['blue', 'orange', 'green', 'red']
for ind_field in range(6):
    for ind in range(4):
        for season in range(6):
            if season == 0:
                plt.plot((field0_observation_times[season*season_fullcycle_obs_times.size:(season+1)*season_fullcycle_obs_times.size] + ind*time_on_each_field)/(365*24*60*60),
                rad_to_nano_as*num_stars**-0.5*magnitudes_no_DVA_no_noise_no_DVA_fitting[ind_field, ind,season*season_fullcycle_obs_times.size:(season+1)*season_fullcycle_obs_times.size],
                    label = labels[ind], color = colors[ind])
            else:
                plt.plot((field0_observation_times[season*season_fullcycle_obs_times.size:(season+1)*season_fullcycle_obs_times.size] + ind*time_on_each_field)/(365*24*60*60),
                rad_to_nano_as*num_stars**-0.5*magnitudes_no_DVA_no_noise_no_DVA_fitting[ind_field, ind,season*season_fullcycle_obs_times.size:(season+1)*season_fullcycle_obs_times.size],
                    color = colors[ind])
    plt.legend()
    plt.xlabel('Time (years)')
    #plt.ylabel('RMS deflection (radians)')
    #plt.ylabel(r'RMS deflection ($\mu$as)')
    plt.ylabel(r'RMS deflection (nas)')
    plt.title('Gradient Modes, no simulated DVA or noise, no fitted DVA removal, field ' + str(ind_field))
    plt.show()
    

In [ ]:
#Plot results with no simulated DVA and no simulated noise.

labels = ['XX', 'XY', 'YY', 'YX']
for ind_field in range(6):
    for ind in range(4):
        plt.plot((field0_observation_times + ind*time_on_each_field)/(365*24*60*60),
        num_stars**-0.5*magnitudes_no_DVA_no_noise_with_DVA_fitting[ind_field, ind,:],
                label = labels[ind])
    plt.legend()
    plt.xlabel('Time (years)')
    plt.ylabel('RMS deflection (radians)')
    plt.title('Gradient Modes, no simulated DVA or noise, fitted DVA removal, field ' + str(ind_field))
    plt.show()

In [ ]:
#Plot results with simulated DVA and no simulated noise.

#for ind_field in range(6):
#    for ind in range(4):
#        plt.plot((field0_observation_times + ind*time_on_each_field)/(365*24*60*60),
#        num_stars**-0.5*magnitudes_with_DVA_no_noise_no_DVA_fitting[ind_field, ind,:],
#                label = labels[ind])
#    plt.legend()
#    plt.xlabel('Time (years)')
#    plt.ylabel('RMS deflection (radians)')
#    plt.title('Gradient Modes, with simulated DVA but no noise, no fitted DVA removal, field ' + str(ind_field))
#    plt.show()

labels = ['XX', 'XY', 'YY', 'YX']
colors = ['blue', 'orange', 'green', 'red']
#to_nas = 
for ind_field in range(6):
    for ind in range(4):
        for season in range(6):
            if season == 0:
                plt.plot((field0_observation_times[season*season_fullcycle_obs_times.size:(season+1)*season_fullcycle_obs_times.size] + ind*time_on_each_field)/(365*24*60*60),
                num_stars**-0.5*magnitudes_with_DVA_no_noise_no_DVA_fitting[ind_field, ind,season*season_fullcycle_obs_times.size:(season+1)*season_fullcycle_obs_times.size],
                    label = labels[ind], color = colors[ind])
            else:
                plt.plot((field0_observation_times[season*season_fullcycle_obs_times.size:(season+1)*season_fullcycle_obs_times.size] + ind*time_on_each_field)/(365*24*60*60),
                num_stars**-0.5*magnitudes_with_DVA_no_noise_no_DVA_fitting[ind_field, ind,season*season_fullcycle_obs_times.size:(season+1)*season_fullcycle_obs_times.size],
                    color = colors[ind])
    plt.legend()
    plt.xlabel('Time (years)')
    plt.ylabel('RMS deflection (radians)')
    plt.title('Gradient Modes, no simulated DVA or noise, no fitted DVA removal, field ' + str(ind_field))
    plt.show()
    

In [ ]:
for ind_field in range(6):
    for ind in range(4):
        plt.plot((field0_observation_times + ind*time_on_each_field)/(365*24*60*60),
        num_stars**-0.5*magnitudes_with_DVA_no_noise_with_DVA_fitting[ind_field, ind,:],
                label = labels[ind])
    plt.title('Gradient Modes, with simulated DVA but no noise, with fitted DVA removal, field ' + str(ind_field))
    plt.legend()
    plt.xlabel('Time (years)')
    plt.ylabel('RMS deflection (radians)')
    plt.show()

In [ ]:
#Plot results with simulated DVA and simulated noise.

for ind_field in range(6):
    for ind in range(4):
        plt.plot((field0_observation_times + ind*time_on_each_field)/(365*24*60*60),
        num_stars**-0.5*magnitudes_with_DVA_with_noise_no_DVA_fitting[ind_field, ind,:],
                label = labels[ind])
    plt.title('Gradient Modes, with simulated DVA and noise, no fitted DVA removal, field ' + str(ind_field))
    plt.legend()
    plt.xlabel('Time (years)')
    plt.ylabel('RMS deflection (radians)')
    plt.show()

In [ ]:
#Plot results with simulated DVA and simulated noise.

for ind_field in range(6):
    for ind in range(4):
        plt.plot((field0_observation_times + ind*time_on_each_field)/(365*24*60*60),
    num_stars**-0.5*magnitudes_with_DVA_with_noise_with_DVA_fitting[ind_field, ind,:],
                label = labels[ind])
    plt.title('Gradient Modes, with simulated DVA and noise, with fitted DVA removal, field ' + str(ind_field))
    plt.legend()
    plt.xlabel('Time (years)')
    plt.ylabel('RMS deflection (radians)')
    plt.show()

In [ ]:
nan_indices = np.where(np.isnan(magnitudes_with_DVA_with_noise_with_DVA_fitting))
for ind in range(nan_indices[0].size):
    print(magnitudes_with_DVA_with_noise_with_DVA_fitting[nan_indices[0][ind],nan_indices[1][ind],nan_indices[2][ind]])
    magnitudes_with_DVA_with_noise_with_DVA_fitting[nan_indices[0][ind],nan_indices[1][ind],nan_indices[2][ind]] = magnitudes_with_DVA_with_noise_with_DVA_fitting[nan_indices[0][ind],nan_indices[1][ind],nan_indices[2][ind]+1]
#print(nan_indices)

In [ ]:
print(magnitudes_no_DVA_no_noise_no_DVA_fitting.shape)

In [ ]:
#Test Lomb-Scargle periodogram.

from stingray.lightcurve import Lightcurve
from stingray.lombscargle import LombScargleCrossspectrum, LombScarglePowerspectrum
import astropy

test_curve = Lightcurve(field0_observation_times, 
                        magnitudes_no_DVA_no_noise_no_DVA_fitting[0,0,:])

test_curve2 = Lightcurve(field0_observation_times, 
                        magnitudes_with_DVA_no_noise_no_DVA_fitting[0,0,:])

test_lsp = LombScarglePowerspectrum(test_curve, min_freq = 0,
                       norm ='abs')
test_lsp2 = LombScarglePowerspectrum(test_curve2, min_freq = 0,
                       norm ='abs')

In [ ]:
print(magnitudes_with_DVA_no_noise_no_DVA_fitting[0,0,:])

In [ ]:
print(np.sum(magnitudes_no_DVA_no_noise_no_DVA_fitting[0,0,:]**2)/np.sum(test_lsp.power))
print(magnitudes_no_DVA_no_noise_no_DVA_fitting[0,0,:].shape)
print(test_lsp.power.shape)

In [ ]:
plt.plot(test_lsp.freq, test_lsp.power)
plt.plot(test_lsp2.freq, test_lsp2.power)
plt.xscale('log')
plt.yscale('log')

In [ ]:
#Write a function to deproject DVA via harmonics of the orbital period.

def make_orthonormal_periodic_basis(times, num_harmonics = 3):
    normal_basis = np.zeros((2*num_harmonics, times.size))
    norm_factor = 2**0.5*times.size**-0.5
    for harmonic in range(1, num_harmonics+1):
        normal_basis[2*harmonic-2,:] = norm_factor*np.cos(2*np.pi*times*harmonic*(1./(365*24*3600.)))
        normal_basis[2*harmonic-1,:] = norm_factor*np.sin(2*np.pi*times*harmonic*(1./(365*24*3600.)))
    svd = np.linalg.svd(normal_basis, full_matrices=False)
    return svd[2]

field0_year_harmonic_basis = make_orthonormal_periodic_basis(field0_observation_times)
#plt.plot(field0_observation_times, field0_year_harmonic_basis[0,:])
#plt.plot(field0_observation_times, field0_year_harmonic_basis[1,:])
#plt.show()

full_obs_year_harmonic_basis = make_orthonormal_periodic_basis(observing_times_seconds)
#plt.plot(observing_times_seconds, full_obs_year_harmonic_basis[0,:])
#plt.plot(observing_times_seconds, full_obs_year_harmonic_basis[1,:])
#plt.show()

def deproject_orthonormal_basis(data, basis):
    #Basis dimension is (num_modes, data_size).
    #Data dimension is (data_size).
    amps = np.matmul(data, basis.T)
    projected_data = np.matmul(amps, basis)
    return data - projected_data

#Test deprojection.

def deproject_harmonics_from_fields_and_templates(data, num_harmonics = 3):
    deprojected_data = np.zeros(data.shape)
    for field_ind in range(data.shape[0]):
        times = field0_observation_times + field_ind*time_on_each_field
        basis = make_orthonormal_periodic_basis(times, 
                                    num_harmonics = num_harmonics)
        for template_ind in range(data.shape[1]):
            deprojected_data[field_ind, template_ind, :] = deproject_orthonormal_basis(data[field_ind, template_ind, :], basis)
    return deprojected_data

plt.plot(field0_observation_times, magnitudes_with_DVA_with_noise_no_DVA_fitting[0, 0,:])
plt.plot(field0_observation_times, 
  deproject_orthonormal_basis(magnitudes_with_DVA_with_noise_no_DVA_fitting[0, 0,:],
                             field0_year_harmonic_basis))
plt.title('Test DVA time-space harmonic de-projection.')
plt.show()


In [ ]:
#De-project harmonic modes of a year from some data.
#This may be necessary to remove DVA.
#For now, we assume perfect knowledge of the orbit.

print(magnitudes_with_DVA_no_noise_no_GW_no_DVA_fitting.shape)
deproj_magnitudes_with_DVA_no_noise_no_GW_no_DVA_fitting = deproject_harmonics_from_fields_and_templates(magnitudes_with_DVA_no_noise_no_GW_no_DVA_fitting, 
                                                num_harmonics=9)

deproj_magnitudes_with_DVA_no_noise_no_GW_with_DVA_fitting = deproject_harmonics_from_fields_and_templates(magnitudes_with_DVA_no_noise_no_GW_with_DVA_fitting, 
                                                num_harmonics=9)


deproj_magnitudes_with_DVA_no_noise_no_DVA_fitting = deproject_harmonics_from_fields_and_templates(magnitudes_with_DVA_no_noise_no_DVA_fitting, 
                                                num_harmonics=9)

deproj_magnitudes_with_DVA_no_noise_with_DVA_fitting = deproject_harmonics_from_fields_and_templates(magnitudes_with_DVA_no_noise_with_DVA_fitting, 
                                                num_harmonics=9)


In [ ]:
#Compute Lomb Scargle Periodogram auto powers.

def compute_LS_autos(data, norm = 'abs'):
    full_ls_list = []
    for field_ind in range(6):
        field_ls_list = []
        for mode_ind in range(4):
            curve = Lightcurve(field0_observation_times + field_ind*time_on_each_field, 
                data[field_ind, mode_ind ,:])
            ls = LombScarglePowerspectrum(curve, min_freq = 0,
                       norm = norm)
            field_ls_list.append(ls)
        full_ls_list.append(field_ls_list)
    return full_ls_list

LS_no_DVA_no_noise_no_DVA_fitting = compute_LS_autos(magnitudes_no_DVA_no_noise_no_DVA_fitting)
LS_no_DVA_no_noise_with_DVA_fitting = compute_LS_autos(magnitudes_no_DVA_no_noise_with_DVA_fitting)

LS_with_DVA_no_noise_no_DVA_fitting = compute_LS_autos(magnitudes_with_DVA_no_noise_no_DVA_fitting)
LS_with_DVA_no_noise_with_DVA_fitting = compute_LS_autos(magnitudes_with_DVA_no_noise_with_DVA_fitting)

LS_with_DVA_with_noise_no_DVA_fitting = compute_LS_autos(magnitudes_with_DVA_with_noise_no_DVA_fitting)
LS_with_DVA_with_noise_with_DVA_fitting = compute_LS_autos(magnitudes_with_DVA_with_noise_with_DVA_fitting)

LS_with_DVA_no_noise_no_GW_no_DVA_fitting = compute_LS_autos(magnitudes_with_DVA_no_noise_no_GW_no_DVA_fitting)
LS_with_DVA_no_noise_no_GW_with_DVA_fitting = compute_LS_autos(magnitudes_with_DVA_no_noise_no_GW_with_DVA_fitting)

LS_with_DVA_no_noise_no_GW_no_DVA_fitting_deproj = compute_LS_autos(deproj_magnitudes_with_DVA_no_noise_no_GW_no_DVA_fitting)
LS_with_DVA_no_noise_no_GW_with_DVA_fitting_deproj = compute_LS_autos(deproj_magnitudes_with_DVA_no_noise_no_GW_with_DVA_fitting)

LS_with_DVA_no_noise_no_DVA_fitting_deproj = compute_LS_autos(deproj_magnitudes_with_DVA_no_noise_no_DVA_fitting)
LS_with_DVA_no_noise_with_DVA_fitting_deproj = compute_LS_autos(deproj_magnitudes_with_DVA_no_noise_with_DVA_fitting)


In [ ]:

for ind in range(4):
    current_LS = LS_no_DVA_no_noise_no_DVA_fitting[1][ind]
    plt.plot(current_LS.freq, (current_LS.freq.size**-2)*(4./num_stars**-1)*np.abs(current_LS.power),
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Field 1 LS, no noise, no DVA, no DVA fitting.')
plt.show()

for ind in range(4):
    current_LS = LS_no_DVA_no_noise_with_DVA_fitting[1][ind]
    plt.plot(current_LS.freq, (current_LS.freq.size**-2)*(4./num_stars**-1)*np.abs(current_LS.power),
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Field 1 LS, no noise, no DVA, with DVA fitting.')
plt.show()

In [ ]:

for ind in range(4):
    current_LS = LS_with_DVA_no_noise_no_DVA_fitting[0][ind]
    plt.plot(current_LS.freq, (current_LS.freq.size**-2)*(4./num_stars**-1)*np.abs(current_LS.power),
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Field 0 LS, no noise, with DVA, no DVA fitting.')
plt.show()

for ind in range(4):
    current_LS = LS_with_DVA_no_noise_with_DVA_fitting[0][ind]
    plt.plot(current_LS.freq, (current_LS.freq.size**-2)*(4./num_stars**-1)*np.abs(current_LS.power),
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Field 0 LS, no noise, with DVA, with DVA fitting.')
plt.show()

In [ ]:

for ind in range(4):
    current_LS = LS_with_DVA_no_noise_no_GW_no_DVA_fitting[0][ind]
    plt.plot(current_LS.freq, (current_LS.freq.size**-2)*(4./num_stars**-1)*np.abs(current_LS.power),
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('LS, no noise, with DVA, no GW, no DVA fitting.')
plt.show()

for ind in range(4):
    current_LS = LS_with_DVA_no_noise_no_GW_with_DVA_fitting[0][ind]
    plt.plot(current_LS.freq, (current_LS.freq.size**-2)*(4./num_stars**-1)*np.abs(current_LS.power),
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('LS, no noise, with DVA, no GW, with DVA fitting.')
plt.show()

In [ ]:

for ind in range(4):
    current_LS = LS_with_DVA_no_noise_no_GW_no_DVA_fitting_deproj[0][ind]
    plt.plot(current_LS.freq, (current_LS.freq.size**-2)*(4./num_stars**-1)*np.abs(current_LS.power),
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('LS, no noise, with DVA, no GW, no DVA fitting, 12 year harmonics removed.')
plt.show()

for ind in range(4):
    current_LS = LS_with_DVA_no_noise_no_GW_with_DVA_fitting_deproj[0][ind]
    plt.plot(current_LS.freq, (current_LS.freq.size**-2)*(4./num_stars**-1)*np.abs(current_LS.power),
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('LS, no noise, with DVA, no GW, with DVA fitting, 12 year harmonics removed..')
plt.show()

for ind in range(4):
    current_LS = LS_with_DVA_no_noise_no_GW_with_DVA_fitting_deproj[0][ind]
    plt.plot(current_LS.freq, rad_to_nano_as**2*(current_LS.freq.size**-2)*(4./num_stars**-1)*np.abs(current_LS.power),
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], rad_to_nano_as**2*2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'GWB expected signal')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (nas$^2$)')
plt.title('Lomb-Scargle PSD, DVA systematics only')
plt.show()



In [ ]:

for ind in range(4):
    current_LS = LS_with_DVA_no_noise_no_DVA_fitting_deproj[0][ind]
    plt.plot(current_LS.freq, (current_LS.freq.size**-2)*(4./num_stars**-1)*np.abs(current_LS.power),
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('LS, no noise, with DVA, no DVA fitting, 12 year harmonics removed.')
plt.show()

for ind in range(4):
    current_LS = LS_with_DVA_no_noise_with_DVA_fitting_deproj[0][ind]
    plt.plot(current_LS.freq, (current_LS.freq.size**-2)*(4./num_stars**-1)*np.abs(current_LS.power),
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('LS, no noise, with DVA, with DVA fitting, 12 year harmonics removed..')
plt.show()

for ind in range(4):
    current_LS = LS_with_DVA_no_noise_with_DVA_fitting_deproj[0][ind]
    plt.plot(current_LS.freq, rad_to_nano_as**2*(current_LS.freq.size**-2)*(4./num_stars**-1)*np.abs(current_LS.power),
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], rad_to_nano_as**2*2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'GWB expected signal')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (nas$^2$)')
plt.title('Lomb-Scargle PSD, with DVA and GWB')
plt.show()

In [ ]:

for ind in range(4):
    current_LS = LS_with_DVA_with_noise_no_DVA_fitting[0][ind]
    plt.plot(current_LS.freq, (current_LS.freq.size**-2)*(4./num_stars**-1)*np.abs(current_LS.power),
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('LS, no noise, with DVA, no DVA fitting.')
plt.show()

for ind in range(4):
    current_LS = LS_with_DVA_with_noise_with_DVA_fitting[0][ind]
    plt.plot(current_LS.freq, (current_LS.freq.size**-2)*(4./num_stars**-1)*np.abs(current_LS.power),
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('LS, with noise, with DVA, with DVA fitting.')
plt.show()

In [ ]:
#Try making a S/N estimate.

print(gwb_object.full_sky_astrometric_gw_variance_dn_freq)
print(gwb_object.drawn_frequencies.size)

print(np.sum((gwb_object.dn_theta_freq_variance**2)/gwb_object.full_sky_astrometric_gw_variance_dn_freq)**0.5)

empirical_var = np.nanvar((LS_with_DVA_with_noise_with_DVA_fitting[0][0].power[50:]))

print(empirical_var)

In [ ]:
print(LS_with_DVA_with_noise_with_DVA_fitting)